# 1 Задача

In [2]:
import pandas as pd
import sqlite3

df = pd.read_csv(r'C:\Users\Gleb\Desktop\Курсы\Стажировки\sputnik8\sputnik.csv')

display(df.head(10))

,event_category,event_action,event_label,total_events,unique_events
0,city_landing,price_button_submit,Tula / Показать предложения (11),8,7
1,city_landing,price_button_submit,Sochi / Показать предложения (39),12,10
2,city_landing,search-tools-button_open,Penza / Сортировка,2,2
3,city_landing,filters-categories_click,Gelendzhik / ЭКСКУРСИИ В ГЕЛЕНДЖИКЕ ЦЕНЫ,1,1
4,city_landing,price_button_submit,Moscow / Показать предложения (345),2,2
5,city_landing,clear_filter_mobile,Vienna,12,11
6,city_landing,start_date_click,Vilnius,9,4
7,city_landing,ticket-type_checkbox,Irkutsk / Мини-группа,29,26
8,city_landing,price_button_submit,Vladivostok / Показать предложения (28),3,3
9,city_landing,filters-categories_click,Krakow / Необычные,3,3


In [3]:
conn = sqlite3.connect(":memory:")
df.to_sql("travels", conn, index=False, if_exists="replace")

19092

## SQL

**1 Задача**

In [123]:
filters = """
WITH cities as (
SELECT event_category,
       event_action,
            UPPER(CASE 
            WHEN INSTR(event_label, '/') > 0
            THEN TRIM(SUBSTR(event_label, 1, INSTR(event_label, '/') - 1))
            ELSE TRIM(event_label) END) AS city,	
       total_events,
       unique_events
       FROM travels
),

kolvo as (
SELECT SUM(unique_events) FILTER(WHERE event_action = 'Page Visit') as unique_pages_visit,
       SUM(unique_events) FILTER(WHERE event_action != 'Page Visit') as unique_filters_visit
FROM cities
)

SELECT *, ROUND(100.0 * unique_filters_visit / unique_pages_visit, 2) as filter_percent FROM kolvo
"""

In [124]:
result1 = pd.read_sql(filters, conn)
display(result1)

,unique_pages_visit,unique_filters_visit,filter_percent
0,1384478,294969,21.31


**2 Задача**

In [157]:
cities = """
WITH cities as (
SELECT event_category,
       event_action,
            UPPER(CASE 
            WHEN INSTR(event_label, '/') > 0
            THEN TRIM(SUBSTR(event_label, 1, INSTR(event_label, '/') - 1))
            ELSE TRIM(event_label) END) AS city,	
       total_events,
       unique_events
       FROM travels
),
city_kolvo AS (
SELECT city,
       SUM(CASE WHEN event_action = 'Page Visit' THEN unique_events ELSE 0 END) AS page_users,
        SUM(CASE WHEN event_action != 'Page Visit' THEN unique_events ELSE 0 END) AS filter_users
FROM cities
GROUP BY 1
)

SELECT *, ROUND(CAST(filter_users AS FLOAT) / page_users, 2) AS filter_rate FROM city_kolvo
WHERE page_users > 0

"""

In [158]:
result2 = pd.read_sql(cities, conn).set_index('city')

# Топ 10 по использованию фильтров
top_cities = result2.sort_values('filter_rate', ascending = False).head(10)

# Топ 10 с наименьшим использованием
bottom_cities = result2.sort_values('filter_rate').head(10)

print('Топ 10 городов по использованию фильтров')
display(top_cities)

print()

print('Топ 10 городов с наименьшим использованием фильтров')
display(bottom_cities)



Топ 10 городов по использованию фильтров


,page_users,filter_users,filter_rate
city,,,
STRESA,2,4,2.00
MIASS,10,13,1.30
NEVYANSK,4,5,1.25
MANAMA,6,6,1.00
AMIENS,5,5,1.00
TARTU,3,3,1.00
LJUBLJANA,124,103,0.83
BOGOLYUBOVO,5,4,0.80
CADIZ,14,11,0.79



Топ 10 городов с наименьшим использованием фильтров


,page_users,filter_users,filter_rate
city,,,
OSLO,30,0,0.0
AOSTA,1,0,0.0
AQABA,21,0,0.0
ARKHANGELSKOYE,17,0,0.0
ARKHIPO-OSIPOVKA,1123,0,0.0
ARONA,4,0,0.0
ARUSHA,3,0,0.0
ARZAMAS,10,0,0.0
AMBOISE,2,0,0.0


**3 Задача**

In [163]:
section = """
WITH base as (
SELECT event_action, unique_events,
CASE 
WHEN event_action IN (
                'ticket-type_checkbox',
                'pay-type_checkbox',
                'dates_filter_mobile',
                'clear_filter_mobile',
                'start_date_click',
                'end_date_click') THEN 'filters'   
WHEN event_action = 'filters-categories_click' THEN 'sorting'
WHEN event_action = 'search-tools-button_open'THEN 'categories'
ELSE 'other'
END AS section
FROM travels
)

SELECT section, SUM(unique_events) AS unique_events FROM base
WHERE event_action != 'Page Visit'
GROUP BY 1
ORDER BY 2 DESC

"""

In [165]:
result3 = pd.read_sql(section, conn).set_index('section')
display(result3)

,unique_events
section,
categories,85773
other,83975
filters,62682
sorting,62539


**4 Задача**

In [168]:
price = """

WITH price_kolvo as (
SELECT SUM(CASE WHEN event_action IN (
                'price_first',
                'price_second',
                'price_third',
                'price_button_submit'
            ) THEN unique_events ELSE 0 END) AS price_usage,
SUM(unique_events) FILTER(WHERE event_action != 'Page Visit') as unique_filters_visit
FROM travels
)
SELECT price_usage, ROUND(100.0 * price_usage / unique_filters_visit, 2)AS price_percent FROM price_kolvo 

"""

In [169]:
result4 = pd.read_sql(price, conn)
display(result4)

,price_usage,price_percent
0,72329,24.52


## Python

**1 Задача**

In [159]:
# Выделяем город (до символа "/")
df["city"] = df["event_label"].str.split("/").str[0].str.strip().str.lower().str.title()

# Определяем события фильтров (все кроме Page Visit)
filter_df = df[df["event_action"] != "Page Visit"]
page_visit_df = df[df["event_action"] == "Page Visit"]

total_page_users = page_visit_df["unique_events"].sum()
total_filter_users = filter_df["unique_events"].sum()

# print(page_visit_df.shape[0])
# print(filter_df.shape[0])

print(f'Всего уникальных посещений страниц городов: {total_page_users}')
print(f'Сумма уникальных событий с фильтрами (может быть с дублями): {total_filter_users}')
print(f'Примерный % пользователей, взаимодействующих с фильтрами: {total_filter_users / total_page_users * 100:.2f}%')

Всего уникальных посещений страниц городов: 1384478
Сумма уникальных событий с фильтрами (может быть с дублями): 294969
Примерный % пользователей, взаимодействующих с фильтрами: 21.31%


**2 Задача**

In [160]:

city_page = page_visit_df[['unique_events', 'city']].groupby(by = 'city').sum()
city_filter = filter_df[['unique_events', 'city']].groupby(by = 'city').sum()

city_stats = city_page.merge(city_filter, on = 'city', how = 'left', suffixes = ('_page', '_filter')).fillna(0)
city_stats['filter_rate'] = (city_stats['unique_events_filter'] / city_stats['unique_events_page']).round(2)

# Топ 10 по использованию фильтров
top_cities = city_stats.sort_values('filter_rate', ascending = False).head(10)

# Топ 10 с наименьшим использованием
bottom_cities = city_stats.sort_values('filter_rate').head(10)

print('Топ 10 городов по использованию фильтров')
display(top_cities)

print()

print('Топ 10 городов с наименьшим использованием фильтров')
display(bottom_cities)



Топ 10 городов по использованию фильтров


,unique_events_page,unique_events_filter,filter_rate
city,,,
Stresa,2,4.0,2.00
Miass,10,13.0,1.30
Nevyansk,4,5.0,1.25
Manama,6,6.0,1.00
Amiens,5,5.0,1.00
Tartu,3,3.0,1.00
Ljubljana,124,103.0,0.83
Bogolyubovo,5,4.0,0.80
Cadiz,14,11.0,0.79



Топ 10 городов с наименьшим использованием фильтров


,unique_events_page,unique_events_filter,filter_rate
city,,,
Oslo,30,0.0,0.0
Aosta,1,0.0,0.0
Aqaba,21,0.0,0.0
Arkhangelskoye,17,0.0,0.0
Arkhipo-Osipovka,1123,0.0,0.0
Arona,4,0.0,0.0
Arusha,3,0.0,0.0
Arzamas,10,0.0,0.0
Amboise,2,0.0,0.0


**3 Задача**

In [151]:
filters_actions = [
    'ticket-type_checkbox',
    'pay-type_checkbox',
    'dates_filter_mobile',
    'clear_filter_mobile',
    'start_date_click',
    'end_date_click'
]

sorting_actions = ['filters-categories_click']
categories_actions = ['search-tools-button_open']

df['section'] = 'other'
df.loc[df['event_action'].isin(filters_actions), 'section'] = 'filters'
df.loc[df['event_action'].isin(sorting_actions), 'section'] = 'sorting'
df.loc[df['event_action'].isin(categories_actions), 'section'] = 'categories'

sections = df[['section', 'unique_events']][df['event_action'] != 'Page Visit'].groupby('section').sum().sort_values(by = 'unique_events', ascending=False)

display(sections)

,unique_events
section,
categories,85773
other,83975
filters,62682
sorting,62539


**4 задача**

In [152]:

price_actions = [
    'price_first',
    'price_second',
    'price_third',
    'price_button_submit'
]

price_usage = df[df["event_action"].isin(price_actions)]['unique_events'].sum()

total_filter_users = filter_df["unique_events"].sum()

print(f'Пользователей ценовых фильтров: {price_usage}')
print(f'Доля от всех пользователей фильтров: { price_usage / total_filter_unique:.2%}')


Пользователей ценовых фильтров: 72329
Доля от всех пользователей фильтров: 24.52%


# 2 Задача

**Главная цель** внедрения фильтра по времени начала (утро / день / вечер) —  упростить и ускорить поиск подходящей экскурсии

Фильтр помогает пользователям быстрее убрать неподходящие варианты  и сфокусироваться на релевантных

Если фильтр работает правильно, то пользователь:

- быстрее находит нужный вариант,
- тратит меньше усилий на поиск,
- чаще доходит до бронирования
---

**Основная метрика:** `Конверсия в бронирование (CR)`

**Вспомогательные метрики**: 

- Конверсия в клик по экскурсии
- Время на странице
- Количество возвратов к списку
- Средний чек (AOV)
- Выручка на пользователя (ARPU)
- Доля отмен
- Доля пользователей, применивших фильтр

---

Расчитаем `конверсию в бронирование для каждой группы`:

**Группа А**: 

- 5000 пользователей  
- 300 бронирований  

**Конверсия**:

CR_A = 300 / 5000 = 6%

---

**Группа B (с фильтром)**

- 5000 пользователей  
- 450 бронирований  

**Конверсия**:

CR_B = 450 / 5000 = 9%

---

**Вывод:**

Конверсия выросла: `с 6% до 9%`

`Абсолютный прирост:`

9% − 6% = 3 процентных пункта

`Относительный рост:`

(9% − 6%) / 6% = 50%

`Прирост составил` **+150 Бронирований**

---

Новый фильтр увеличил конверсию `с 6% до 9%`, что соответствует росту на `3` процентных пункта или на `50%` относительно контрольной группы. Это существенный прирост и дополнительно `150` бронирований на 5000 пользователей.  

При подтверждении статистической значимости с помощью `z-теста для пропорций` и отсутствии негативного влияния на выручку и другие ключевые метрики фильтр целесообразно внедрять

---

**Проверим статистикой:**

Группа A  
n₁ = 5000  
x₁ = 300  
p₁ = 300 / 5000 = 0.06  

Группа B  
n₂ = 5000  
x₂ = 450  
p₂ = 450 / 5000 = 0.09  

Разница пропорций  

Δp = p₂ − p₁ = 0.09 − 0.06 = 0.03  

---

`Z-тест для пропорций`

**Нулевая гипотеза**

H₀: p₁н = p₂н  
H₁: p₁н ≠ p₂н  

---

**Оценка стандартной ошибки**

ESE = √[ (p1(1 − p1) / n1) + (p2(1 - p2)/ n₂) ]

Подробнее про формулу расчета z-статистики для сравнения пропорций можно узнать здесь:  
https://practicum.yandex.ru/trainer/statistics-basic/lesson/3f893114-03cc-4581-9140-42ede5b8984b/

p1(1 − p1) / n1 = 0.06 × 0.94 / 5000 = 0.0564 / 5000 = 0.00001128

p2(1 − p2) / n1 = 0.09 × 0.91 / 5000 = 0.0819 / 5000 = 0.00001638

ESE = √[0.00001128 + 0.00001638] = 0.0052516664

---

**Z статистика**

Z = Δp / ESE  = 0.03 / 0.0052516664 = 5.7124725211

Наше значение не попадает в интервал `от −1.96 до 1.96`, то есть выходит за пределы допустимой области. Это означает, что результат является достаточно экстремальным, и у нас есть основания отвергнуть нулевую гипотезу

---

**P-value**

- Z = 5.71 
- p value < 0.05  

Разница статистически значима


# 3 Задача

Перед нами `4` карточки:

**A  F  2  7**

Правило:

> Если на одной стороне карточки — гласная буква,  
> то на другой стороне этой карточки должно быть четное число

---

`Английские буквы`

**Гласные:** A, E, I, O, U  

**Согласные:** B, C, D, F, G, H, J, K, L, M, N, P, Q, R, S, T, V, W, X, Y, Z  

---

**Теперь проверяем каждую карточку**

---

**A**

Это гласная. Нужно проверить, есть ли на обратной стороне четное число: `2, 4, 6...`  

Если окажется `7, 5, 3...` — правило нарушено  

Поэтому ее обязательно переворачиваем

---

**F**

Это согласная. А у нас правило ничего не говорит о согласных. У нас может быть как `четное`, так и `нечетное` число на обороте данной карточки

Переворачивать не нужно

---

**2**

Это четное число.  

Правило гласит: "Если на одной стороне карточки  гласная буква, то на другой стороне этой карточки должно быть четное число"

Но в правиле ничего нет о том, что четные числа должны быть на тех карточках, у которых на обратной стороне гласная буква 

Предположим, что на другой стороне карточки с цифрой 2 будет буква `D`, `T` и т.д. . Это не противоречит правилу. Т.е. нам неважно, какая там будет буква: `гласная` или `согласная`

Переворачивать не нужно.

---

**7**

Это нечетное число  

Нужно перевернуть еще и эту карточку, поскольку может получиться так, что на ее оборотной стороне окажется `гласная` буква, а это будет противоречить правилу.

Нужно проверить отсутствие гласной, поэтому ее обязательно переворачиваем

---

**Итог**

Перевернуть нужно: `A и 7` 
